In [3]:
from pathlib import Path
import sys

sys.path.append(str(Path("..").resolve()))

from src.tse_io import (
    list_zip_files,
    read_brasil_file,
    parse_money,
    filter_by_cargo,
    filter_by_candidate_name,
    filter_presidential_candidates
)

from src.graph_builder import build_campaign_graph
import networkx as nx

In [5]:
zip_path = Path("data/raw/prestacao_de_contas_eleitorais_candidatos_2022.zip")
files = list_zip_files(zip_path)
files[:10]

['despesas_contratadas_candidatos_2022_BA.csv',
 'despesas_contratadas_candidatos_2022_AP.csv',
 'receitas_candidatos_2022_BRASIL.csv',
 'despesas_contratadas_candidatos_2022_AL.csv',
 'despesas_contratadas_candidatos_2022_AM.csv',
 'despesas_contratadas_candidatos_2022_AC.csv',
 'despesas_contratadas_candidatos_2022_CE.csv',
 'despesas_pagas_candidatos_2022_BA.csv',
 'despesas_contratadas_candidatos_2022_ES.csv',
 'despesas_contratadas_candidatos_2022_GO.csv']

In [ ]:
receitas = read_brasil_file(
    zip_path,
    "receitas_candidatos_2022"
)

despesas_contratadas = read_brasil_file(
    zip_path,
    "despesas_contratadas_candidatos_2022"
)

receitas_originario = read_brasil_file(
    zip_path,
    "receitas_candidatos_doador_originario_2022"
)

In [ ]:
receitas_presidente = filter_by_cargo(receitas, "Presidente")

In [5]:
receitas_pres = filter_presidential_candidates(receitas)
despesas_pres = filter_presidential_candidates(despesas_contratadas)

In [9]:
receitas_pres[["NR_CANDIDATO", "NM_CANDIDATO", "SG_PARTIDO"]].drop_duplicates()

,NR_CANDIDATO,NM_CANDIDATO,SG_PARTIDO
1534,22,JAIR MESSIAS BOLSONARO,PL
21188,13,LUIZ INÁCIO LULA DA SILVA,PT


In [10]:
despesas_pres[["NR_CANDIDATO", "NM_CANDIDATO", "SG_PARTIDO"]].drop_duplicates()

,NR_CANDIDATO,NM_CANDIDATO,SG_PARTIDO
558517,13,LUIZ INÁCIO LULA DA SILVA,PT
563482,22,JAIR MESSIAS BOLSONARO,PL


In [8]:
sq_receitas_pres = receitas_pres["SQ_RECEITA"].dropna().unique()

receitas_originario_pres = receitas_originario[
    receitas_originario["SQ_RECEITA"].isin(sq_receitas_pres)
].copy()

In [ ]:
receitas_com_originario = receitas_pres.merge(
    receitas_originario_pres,
    on="SQ_RECEITA",
    how="left",
    suffixes=("", "_ORIGINARIO")
)

In [ ]:
receitas_com_originario[
    [
        "SQ_RECEITA",
        "NM_CANDIDATO",
        "SG_PARTIDO",
        "DS_FONTE_RECEITA",
        "DS_ORIGEM_RECEITA",
        "NM_DOADOR",
        "NM_DOADOR_ORIGINARIO",
        "DS_CNAE_DOADOR_ORIGINARIO",
        "VR_RECEITA",
    ]
].head()

,SQ_RECEITA,NM_CANDIDATO,SG_PARTIDO,DS_FONTE_RECEITA,DS_ORIGEM_RECEITA,NM_DOADOR,NM_DOADOR_ORIGINARIO,DS_CNAE_DOADOR_ORIGINARIO,VR_RECEITA
0,46245435,JAIR MESSIAS BOLSONARO,PL,OUTROS RECURSOS,Recursos de partido político,DIREÇÃO NACIONAL,NaN,NaN,"2412,88"
1,46245435,JAIR MESSIAS BOLSONARO,PL,OUTROS RECURSOS,Recursos de partido político,DIREÇÃO NACIONAL,NaN,NaN,"1532,45"
2,46245435,JAIR MESSIAS BOLSONARO,PL,OUTROS RECURSOS,Recursos de partido político,DIREÇÃO NACIONAL,NaN,NaN,"1532,45"
3,46245435,JAIR MESSIAS BOLSONARO,PL,OUTROS RECURSOS,Recursos de partido político,DIREÇÃO NACIONAL,NaN,NaN,"1718,87"
4,46245435,JAIR MESSIAS BOLSONARO,PL,OUTROS RECURSOS,Recursos de partido político,DIREÇÃO NACIONAL,NaN,NaN,"827,87"


In [6]:
receitas_com_originario["VR_RECEITA_NUM"] = parse_money(
    receitas_com_originario["VR_RECEITA"]
)

despesas_pres["VR_DESPESA_CONTRATADA_NUM"] = parse_money(
    despesas_pres["VR_DESPESA_CONTRATADA"]
)

NameError: name 'receitas_com_originario' is not defined

In [15]:
(
    receitas_pres
    .assign(VR_RECEITA_NUM=parse_money(receitas_pres["VR_RECEITA"]))
    .groupby(["NR_CANDIDATO", "NM_CANDIDATO", "SG_PARTIDO"], as_index=False)
    .agg(
        total_receita=("VR_RECEITA_NUM", "sum"),
        qtd_receitas=("SQ_RECEITA", "nunique")
    )
    .sort_values("total_receita", ascending=False)
)

,NR_CANDIDATO,NM_CANDIDATO,SG_PARTIDO,total_receita,qtd_receitas
0,13,LUIZ INÁCIO LULA DA SILVA,PT,1.355393e+08,382
1,22,JAIR MESSIAS BOLSONARO,PL,1.261275e+08,392012


In [16]:
(
    despesas_pres
    .assign(VR_DESPESA_CONTRATADA_NUM=parse_money(despesas_pres["VR_DESPESA_CONTRATADA"]))
    .groupby(["NR_CANDIDATO", "NM_CANDIDATO", "SG_PARTIDO"], as_index=False)
    .agg(
        total_despesa_contratada=("VR_DESPESA_CONTRATADA_NUM", "sum"),
        qtd_despesas=("SQ_DESPESA", "nunique")
    )
    .sort_values("total_despesa_contratada", ascending=False)
)

,NR_CANDIDATO,NM_CANDIDATO,SG_PARTIDO,total_despesa_contratada,qtd_despesas
0,13,LUIZ INÁCIO LULA DA SILVA,PT,1.313130e+08,963
1,22,JAIR MESSIAS BOLSONARO,PL,8.906076e+07,672


In [17]:
(
    receitas_pres
    .assign(VR_RECEITA_NUM=parse_money(receitas_pres["VR_RECEITA"]))
    .groupby(["NM_CANDIDATO", "NM_DOADOR", "DS_CNAE_DOADOR"], as_index=False)
    .agg(total_receita=("VR_RECEITA_NUM", "sum"))
    .sort_values("total_receita", ascending=False)
    .head(20)
)

,NM_CANDIDATO,NM_DOADOR,DS_CNAE_DOADOR,total_receita
361202,LUIZ INÁCIO LULA DA SILVA,Direção Nacional,Atividades de organizações políticas,1.243279e+08
81017,JAIR MESSIAS BOLSONARO,DIREÇÃO NACIONAL,Atividades de organizações políticas,3.669334e+07
361311,LUIZ INÁCIO LULA DA SILVA,UM A MAIS SERVIÇOS DE TECNOLOGIA E CONSULTORIA,"Suporte técnico, manutenção e outros serviços ...",6.814847e+06
109208,JAIR MESSIAS BOLSONARO,FABIANO CAMPOS ZETTEL,#NULO,3.000000e+06
189767,JAIR MESSIAS BOLSONARO,JOSÉ SALIM MATTAR JUNIOR,#NULO,1.800000e+06
145527,JAIR MESSIAS BOLSONARO,HUGO DE CARVALHO RIBEIRO,#NULO,1.200000e+06
361200,LUIZ INÁCIO LULA DA SILVA,Direção Estadual/Distrital,Atividades de organizações políticas,1.166137e+06
212504,JAIR MESSIAS BOLSONARO,LUCIANO HANG,#NULO,1.022000e+06
284091,JAIR MESSIAS BOLSONARO,PEDRO GRENDENE BARTELLE,#NULO,1.000000e+06
273621,JAIR MESSIAS BOLSONARO,OSCAR LUIZ CERVI,#NULO,1.000000e+06


In [18]:
(
    receitas_com_originario
    .groupby(
        [
            "NM_CANDIDATO",
            "NM_DOADOR_ORIGINARIO",
            "TP_DOADOR_ORIGINARIO",
            "DS_CNAE_DOADOR_ORIGINARIO",
        ],
        as_index=False,
    )
    .agg(total_receita=("VR_RECEITA_NUM", "sum"))
    .sort_values("total_receita", ascending=False)
    .head(20)
)

,NM_CANDIDATO,NM_DOADOR_ORIGINARIO,TP_DOADOR_ORIGINARIO,DS_CNAE_DOADOR_ORIGINARIO,total_receita
35317,LUIZ INÁCIO LULA DA SILVA,JOSE CARLOS PINHEIRO PRIOSTE,F,#NULO,9182279.41
3839,JAIR MESSIAS BOLSONARO,MAURO FIDELIX DA SILVA,F,#NULO,4791932.34
54399,LUIZ INÁCIO LULA DA SILVA,PAULO AVELINO BARBOSA SILVA,F,#NULO,4239326.76
51111,LUIZ INÁCIO LULA DA SILVA,MIGUEL LEONEL DOS SANTOS,F,#NULO,3403540.38
28465,LUIZ INÁCIO LULA DA SILVA,GILBERTO MAIA,F,#NULO,3150083.17
3735,JAIR MESSIAS BOLSONARO,MARIO ALBERTO MARCHINI,F,#NULO,3064476.91
63986,LUIZ INÁCIO LULA DA SILVA,TARCISIO PRACIANO PEREIRA,F,#NULO,3040936.12
26063,LUIZ INÁCIO LULA DA SILVA,FERNANDO MORENO,F,#NULO,2954110.74
47619,LUIZ INÁCIO LULA DA SILVA,MARIA ISABEL COSTA DA SILVEIRA,F,#NULO,2952258.82
5047,JAIR MESSIAS BOLSONARO,THIAGO VINICIUS TEIXEIRA PEREIRA,F,#NULO,2738564.05


In [19]:
(
    despesas_pres
    .assign(VR_DESPESA_CONTRATADA_NUM=parse_money(despesas_pres["VR_DESPESA_CONTRATADA"]))
    .groupby(
        ["NM_CANDIDATO", "NM_FORNECEDOR", "DS_CNAE_FORNECEDOR", "DS_ORIGEM_DESPESA"],
        as_index=False,
    )
    .agg(total_despesa=("VR_DESPESA_CONTRATADA_NUM", "sum"))
    .sort_values("total_despesa", ascending=False)
    .head(20)
)

,NM_CANDIDATO,NM_FORNECEDOR,DS_CNAE_FORNECEDOR,DS_ORIGEM_DESPESA,total_despesa
533,LUIZ INÁCIO LULA DA SILVA,M4 COMUNICAÇÃO E PROPAGANDA LTDA,Agências de publicidade,"Produção de programas de rádio, televisão ou v...",37899000.00
148,JAIR MESSIAS BOLSONARO,GOOGLE BRASIL INTERNET LTDA.,"Portais, provedores de conteúdo e outros servi...",Despesa com Impulsionamento de Conteúdos,27828538.79
473,LUIZ INÁCIO LULA DA SILVA,GOOGLE BRASIL INTERNET LTDA,"Portais, provedores de conteúdo e outros servi...",Despesa com Impulsionamento de Conteúdos,22933534.93
339,JAIR MESSIAS BOLSONARO,RM FILMES E PUBLICIDADE LTDA,"Atividades de produção cinematográfica, de víd...","Produção de programas de rádio, televisão ou v...",10389869.09
101,JAIR MESSIAS BOLSONARO,DIREÇÃO NACIONAL,Atividades de organizações políticas,Doações financeiras a outros candidatos/partidos,5469206.92
123,JAIR MESSIAS BOLSONARO,FACEBOOK SERVICOS ONLINE DO BRASIL LTDA.,"Agenciamento de espaços para publicidade, exce...",Despesa com Impulsionamento de Conteúdos,5182800.00
373,JAIR MESSIAS BOLSONARO,UNIPAUTA FORMULARIOS LTDA,Impressão de material de segurança,Publicidade por materiais impressos,4974100.00
535,LUIZ INÁCIO LULA DA SILVA,MAGNI E AR PRODUCOES E SHOWS LTDA,"Artes cênicas, espetáculos e atividades comple...",Eventos de promoção da candidatura,4915657.00
446,LUIZ INÁCIO LULA DA SILVA,Direção Estadual/Distrital,Atividades de organizações políticas,Doações financeiras a outros candidatos/partidos,4765500.00
255,JAIR MESSIAS BOLSONARO,MAGIC BEANS COMUNICACAO LTDA,Agências de publicidade,Serviços prestados por terceiros,4419900.00


In [20]:
(
    receitas_pres
    .assign(VR_RECEITA_NUM=parse_money(receitas_pres["VR_RECEITA"]))
    .groupby(
        ["NM_CANDIDATO", "DS_FONTE_RECEITA"],
        as_index=False
    )
    .agg(total=("VR_RECEITA_NUM", "sum"))
    .sort_values("total", ascending=False)
)

,NM_CANDIDATO,DS_FONTE_RECEITA,total
3,LUIZ INÁCIO LULA DA SILVA,FUNDO ESPECIAL,1.251190e+08
2,JAIR MESSIAS BOLSONARO,OUTROS RECURSOS,9.469116e+07
1,JAIR MESSIAS BOLSONARO,FUNDO PARTIDARIO,2.944739e+07
5,LUIZ INÁCIO LULA DA SILVA,OUTROS RECURSOS,1.017819e+07
0,JAIR MESSIAS BOLSONARO,FUNDO ESPECIAL,1.988973e+06
4,LUIZ INÁCIO LULA DA SILVA,FUNDO PARTIDARIO,2.420922e+05


In [21]:
(
    receitas_pres
    .assign(VR_RECEITA_NUM=parse_money(receitas_pres["VR_RECEITA"]))
    .groupby(
        ["NM_CANDIDATO", "DS_ORIGEM_RECEITA"],
        as_index=False
    )
    .agg(total=("VR_RECEITA_NUM", "sum"))
    .sort_values("total", ascending=False)
)

,NM_CANDIDATO,DS_ORIGEM_RECEITA,total
6,LUIZ INÁCIO LULA DA SILVA,Recursos de partido político,1.258297e+08
3,JAIR MESSIAS BOLSONARO,Recursos de pessoas físicas,8.815575e+07
2,JAIR MESSIAS BOLSONARO,Recursos de partido político,3.710355e+07
5,LUIZ INÁCIO LULA DA SILVA,Recursos de Financiamento Coletivo,6.814847e+06
7,LUIZ INÁCIO LULA DA SILVA,Recursos de pessoas físicas,2.829703e+06
0,JAIR MESSIAS BOLSONARO,Recursos de Financiamento Coletivo,5.361398e+05
4,JAIR MESSIAS BOLSONARO,Rendimentos de aplicações financeiras,3.280258e+05
8,LUIZ INÁCIO LULA DA SILVA,Rendimentos de aplicações financeiras,6.500255e+04
1,JAIR MESSIAS BOLSONARO,Recursos de outros candidatos,4.060000e+03


In [7]:
G = build_campaign_graph(
    receitas_com_originario=receitas_com_originario,
    despesas_pres=despesas_pres,
)

G.number_of_nodes(), G.number_of_edges()

NameError: name 'receitas_com_originario' is not defined